# M5S4 · Group project workshop (Session 24)
Turn a project idea into a feasible HPC experiment. This is a planning and evidence-analysis notebook, not a finished student application.

**Learning objectives:** define a correctness rule; choose two meaningful main HPC approaches plus a distinct additional technology experiment; design a fair baseline/scaling/sensitivity comparison; identify a bottleneck; prepare a reproducible handover and justified scale-up estimate.

**Today, 80 minutes:** opening 0–10; scope and architecture 10–30; experiments and diagnosis 30–55; handover and feedback 55–70; peer review and exit 70–80.

**Sources:** official 2026–27 teaching plan, Session 24 p.16 and Session 25 p.17; supplied current Blackboard group assignment; course introduction. Blackboard holds official dates and submission rules. The pasted assignment contains a stale “before session 11” instruction; do not infer a new deadline from this notebook.


## 0 · Run on SciTech
Open JupyterLab at https://jupyter.iesci.tech:8085/, then **File → Open from URL**, paste this notebook’s raw URL and save your own copy. Run Python 3 cells in order. All notebook calculations are tiny, use the standard library and require no allocation or package installation.

The workshop does **not** submit jobs. Run your group’s actual scientific workloads through separate Slurm jobs. Reuse the [cluster guide](../../cluster/README.md) and earlier session notebooks for the exact site environment. Inspect `sinfo`, the available compiler/modules and your current resource limits. No account or QoS is added here. Multi-node work is conditional on availability and relevance; GPU extensions are optional. A GPU or MPI capability is NOT RUN until your own job provides evidence.

Use a student-owned directory. The following cell writes only small workshop files in a new folder. Repeated execution uses the same folder within this kernel.


In [ ]:
from pathlib import Path
import csv, json, math, statistics, sys, tempfile
if 'WORK' not in globals():
    WORK = Path(tempfile.mkdtemp(prefix='m5s4_workshop_', dir=Path.cwd()))
STATUS = {}
print('Workshop files:', WORK)
print('Python:', sys.version.split()[0])
print('Cluster scientific jobs: NOT RUN (this notebook does not submit them).')


## W1 · Define a testable project
**Real problem:** “make a heat solver faster” is too vague to test.
**Prediction:** which decisions must be fixed before comparing two implementations?
**Code / Run:** edit the following planning fields and run the cell. The initial status is deliberately DRAFT.
**Result / Explanation:** the checker reports missing decisions, not scientific success.
**Student experiment:** replace the placeholders with your group’s decisions. Choose a small CPU route that remains feasible if GPUs are unavailable.
**Takeaway:** a project needs a defined input, correctness rule and performance metric.


In [ ]:
plan = {
    'group': '', 'problem': '', 'input_and_size': '',
    'correctness_rule': '', 'baseline': '', 'metric': '',
    'main_approach_1': '', 'main_approach_2': '',
    'additional_technology': '', 'resource_fallback': '',
    'next_action': '', 'owner': ''
}
# Fill the values above. A route such as MPI + OpenMP should explain each role.
missing = [k for k, v in plan.items() if not str(v).strip()]
STATUS['W1_plan'] = 'DRAFT' if missing else 'READY FOR TEACHER REVIEW'
(WORK / 'project_plan.json').write_text(json.dumps(plan, indent=2)+'\n')
print(STATUS['W1_plan'])
print('Decisions still needed:', ', '.join(missing) or 'none')
print('Completeness is not proof of feasibility or correctness.')


### Technology choices
The assignment requires **two meaningful main HPC approaches** and **one distinct additional technology experiment**. For example, MPI distributes subdomains and OpenMP parallelises work within a subdomain; a small OpenACC port can be the additional experiment if supported. An accelerated library/framework route is also possible, but describe what each component contributes. Merely naming two tools does not demonstrate two approaches. Slurm and a container support execution/reproducibility and should not replace the required computational approaches.

A negative additional-technology result is acceptable when you actually tested it and explain the evidence. Unavailable hardware is SKIPPED, not a tested negative result.

**Checkpoint 1:** A group lists “Slurm + container” as its two HPC programming approaches. Best feedback?
A. The list is complete. B. Request a GPU regardless of application. C. Explain two computational approaches and keep Slurm/container as support. D. Replace measurements with architecture screenshots.
<details><summary>Expected reasoning</summary>C. A confuses infrastructure with computation. B ignores scope and availability. D provides no performance or correctness evidence.</details>


## W2 · Analyse a controlled scaling experiment
**Real problem:** the shortest single timing may be noise or a different workload.
**Prediction:** if a fixed workload needs 12 s on one thread and 4.5 s on four threads, is efficiency 100%?
**Code / Run:** analyse the explicitly illustrative timing records below. These are NOT SciTech measurements.
**Result / Explanation:** use the median of three repeated elapsed times. Speedup = T1/Tp and efficiency = speedup/p, with p the comparable CPU-thread count. The fixed problem, algorithm, correctness rule and timing region must match. Queue wait is excluded.
**Student experiment:** change only the eight-thread elapsed times and rerun; explain the resulting efficiency.
**Takeaway:** speedup requires a justified baseline and comparable work.


In [ ]:
# ILLUSTRATIVE teaching data, not results from a cluster run.
records = {1: [12.0, 12.2, 11.8], 2: [6.8, 7.0, 6.9],
           4: [4.4, 4.5, 4.6], 8: [4.1, 4.2, 4.3]}
medians = {p: statistics.median(ts) for p, ts in records.items()}
assert all(len(ts) >= 3 and all(math.isfinite(t) and t > 0 for t in ts)
           for ts in records.values())
t1 = medians[1]
rows = []
print('ILLUSTRATIVE DATA | fixed problem, elapsed seconds, median of 3')
print('threads  median_s  speedup  efficiency')
for p in sorted(medians):
    speedup = t1 / medians[p]
    efficiency = speedup / p
    rows.append({'threads': p, 'median_s': medians[p],
                 'speedup': speedup, 'efficiency': efficiency,
                 'evidence': 'ILLUSTRATIVE'})
    print(f'{p:7d} {medians[p]:9.2f} {speedup:8.2f} {efficiency:10.1%}')
assert math.isclose(rows[2]['speedup'], 12/4.5)
assert math.isclose(rows[2]['efficiency'], 2/3)
with (WORK / 'illustrative_scaling.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
STATUS['W2_metric_calculation'] = 'PASS'
print('PASS: metric calculations. Real group scaling measurements: NOT RUN.')


### Your experiment matrix
Write at least four planned comparisons in a group-owned document: baseline; fixed-workload scaling; one application/system sensitivity parameter; and before/after optimization. State what stays fixed, what changes, the numerical check, repetitions, resources and evidence filename. Start with 1, 2 and 4 threads or another small justified set. Do not copy a 64-core sweep without checking resources.

**Checkpoint 2:** A 4-thread run is twice as fast but uses half the grid cells. Can it support a fixed-workload speedup claim?
A. Yes, because elapsed time decreased. B. No, repeat with the same workload. C. Yes, because threads guarantee equal work. D. No parallel experiment is valid.
<details><summary>Expected reasoning</summary>B. A and C ignore changed work; D is too broad. A changed-size run can be useful but must be labelled and analysed separately.</details>


## W3 · Diagnose, optimize and recheck correctness
**Real problem:** optimizing a small fraction of runtime may have little total effect.
**Prediction:** if compute uses 30%, communication 50% and I/O 20%, how much does doubling compute speed improve the total?
**Code / Run:** apply a simple additive model with no overlap to explicitly illustrative phase times.
**Result / Explanation:** phase timers suggest a hypothesis; validate it with an appropriate profiler or instrumentation on real runs. Recheck correctness before accepting a speedup.
**Student experiment:** change only `improve_phase` to communication; explain the difference. Then change the error to exceed tolerance.
**Takeaway:** optimize the measured bottleneck, and reject an incorrect faster version.


In [ ]:
# ILLUSTRATIVE additive model, not a profiler trace or measured speedup.
phases = {'compute': 3.0, 'communication': 5.0, 'io': 2.0}
improve_phase = 'compute'
phase_speedup = 2.0
assert improve_phase in phases and math.isfinite(phase_speedup) and phase_speedup >= 1
before = sum(phases.values())
after = sum(t / phase_speedup if name == improve_phase else t
            for name, t in phases.items())
print(f'ILLUSTRATIVE model: {before:.1f} s -> {after:.1f} s; speedup {before/after:.3f}')
reference = [1.0, 2.0, 3.0]
candidate = [1.0, 2.0, 3.0 + 1e-13]
tolerance = 1e-12
max_error = max(abs(a-b) for a,b in zip(reference, candidate))
correct = len(reference) == len(candidate) and all(
    math.isclose(a,b,rel_tol=tolerance,abs_tol=tolerance)
    for a,b in zip(reference,candidate))
print('Correctness comparison:', 'PASS' if correct else 'FAIL', 'max error', max_error)
assert math.isclose(before, 10) and after > 0
STATUS['W3_model_and_comparison'] = 'PASS' if correct else 'FAIL'
print('Model verified locally. Group profiling and optimization: NOT RUN.')


**Checkpoint 3:** The optimized version is faster but violates the numerical tolerance. What should the report conclude?
A. Accept it because speed is the goal. B. Hide the check. C. Report the failed check, fix or reject the optimization, then remeasure. D. Increase tolerance until it passes without scientific justification.
<details><summary>Expected reasoning</summary>C. Timing does not validate science. A and B discard correctness. D changes the acceptance rule without a justified error budget.</details>


## W4 · Budget a proposed scale-up
**Real problem:** a larger resource request needs a traceable estimate.
**Prediction:** do 8 nodes × 2 hours × 20 runs require 16 or 320 node-hours?
**Code / Run:** calculate a proposed budget. These numbers are a scenario, not measured EuroHPC performance.
**Result / Explanation:** do not extrapolate perfect scaling from a few teaching-node measurements. Justify target hardware, memory, data movement, software and support needs. GPU-hours and node-hours are different units.
**Student experiment:** change only the number of runs; explain the impact on cost. Then replace the scenario with your justified estimate.
**Takeaway:** state assumptions and units with the resource request.


In [ ]:
nodes, hours_per_run, runs = 8, 2.0, 20
assert nodes > 0 and hours_per_run > 0 and runs > 0
node_hours = nodes * hours_per_run * runs
print(f'PROPOSED SCENARIO: {nodes} nodes x {hours_per_run:g} h/run x {runs} runs = {node_hours:g} node-hours')
assert node_hours == nodes * hours_per_run * runs
STATUS['W4_budget_arithmetic'] = 'PASS'
print('PASS: budget arithmetic. Larger-system execution and allocation approval: NOT RUN.')


## 5 · Handover and peer review
Repository convention from the assignment:
```
src/       env/       slurm/       data/       results/       docs/
README.md  REPRODUCE.md
```
Keep code, input identity, commit, compiler/flags, versions, exact commands, Slurm resources and numerical checks. Keep raw repeated timings and job logs. A result without a completed numerical check is not a validated performance result.

Final assignment: repository; working application; 5–7 page technical report including figures; 1–2 page EuroHPC scale-up plan; approximately five slides for a five-minute presentation. This workshop’s output is a plan, experiment matrix, owners and next action, not the final application.

**Peer review:** can another group explain your calculation, identify the baseline, reproduce the planned small run and state what would falsify the performance claim?

**Checkpoint 4:** A tested GPU prototype gives no speedup. What should you do?
A. Delete it from the technology matrix. B. Invent a larger-GPU result. C. Record the negative result and explain likely limits and supporting evidence. D. Mark an unavailable GPU as a measured negative result.
<details><summary>Expected reasoning</summary>C. A loses useful evidence; B invents results; D confuses SKIPPED with a tested outcome.</details>

**AI policy:** the supplied course introduction permits research, ideation, outlines and proofreading with acknowledgement, but prohibits GenAI-generated final code, answers or group-work contributions unless explicitly authorised. This teaching notebook does not grant an exception.


In [ ]:
print('WORKSHOP CHECKS')
for case, state in STATUS.items():
    print(f'{case:32s} {state}')
print('PASS = this small executed check passed; FAIL = attempted check failed.')
print('SKIPPED = optional/unavailable; NOT RUN = no execution evidence.')
print('DRAFT / READY FOR TEACHER REVIEW are planning states, not execution outcomes.')
print('Group baseline / parallel / GPU / MPI jobs: NOT RUN by this workbook.')
print('Bring the plan and experiment matrix to Session 25; final presentation is Session 29.')


## Validation boundary
The instructor tested all standard-library cells locally, including the illustrative metric/model calculations and file generation. No SciTech allocation, application, MPI, OpenMP, GPU or profiler execution is claimed for this workshop workbook. Students must replace illustrative data with their own measured evidence for the assessed project. A planning completeness check is not teacher approval.

Primary references: official syllabus Session 24 p.16 and Session 25 p.17; current supplied group-assignment text; course-introduction project timeline and AI policy. Earlier sessions provide the tested implementation demonstrations; reuse relevant code only within course assessment rules.
